In [5]:
import pandas as pd
import optuna
from sklearn.metrics import roc_auc_score
import default_risk.config as cfg

y_true = pd.read_parquet(cfg.MASTER_DATA_DIR / 'prepared_dataset_train.parquet')['target']
oof_xgb = pd.read_csv(cfg.ARTIFACTS_DIR / "Parent_Pipeline_xgb-1.1_oof_predictions.csv")['oof_prediction']
oof_lgbm = pd.read_csv(cfg.ARTIFACTS_DIR / "Parent_Pipeline_lgbm-1.1_oof_predictions.csv")['oof_prediction']

def objective(trial):
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    blended = (w_xgb * oof_xgb) + ((1.0 - w_xgb) * oof_lgbm)
    return roc_auc_score(y_true, blended)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

w_xgb_opt = study.best_params['w_xgb']
print(f"Best AUC: {study.best_value:.5f}")
print(f"Weight XGB: {w_xgb_opt:.4f}")
print(f"Weight LGBM: {1.0 - w_xgb_opt:.4f}")

Best AUC: 0.79932
Weight XGB: 0.3352
Weight LGBM: 0.6648
